In [1]:
import os
import json
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm.notebook import tqdm

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [2]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/lasse/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/lasse/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
INPUT_FILE = "995,000_rows.csv"
OUTPUT_DIR = Path("processed_chunks")
OUTPUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 10000
TEXT_COL = "content"
LABEL_COL = "type"

In [4]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize_text(text):
    text = str(text).lower()
    return word_tokenize(text)

def clean_tokens(tokens):
    # Keep alphabetic tokens only, remove stopwords
    return [t for t in tokens if t.isalpha() and t not in stop_words]

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

In [5]:
stats = {
    "n_rows": 0,
    "n_missing_content": 0,
    "total_tokens_before": 0,
    "total_tokens_after_stopwords": 0,
    "unique_tokens_before": set(),
    "unique_tokens_after_stopwords": set(),
    "unique_tokens_after_stemming": set(),
    "top_words_before": Counter(),
    "top_words_after_stopwords": Counter(),
    "top_words_after_stemming": Counter(),
}

In [6]:
chunk_files = []

reader = pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)

for i, chunk in enumerate(tqdm(reader, desc="Processing chunks")):
    # Keep columns if present
    keep_cols = [col for col in [TEXT_COL, LABEL_COL] if col in chunk.columns]
    chunk = chunk[keep_cols].copy()

    #content handling
    missing_mask = chunk[TEXT_COL].isna()
    stats["n_missing_content"] += int(missing_mask.sum())
    chunk = chunk.dropna(subset=[TEXT_COL]).copy()

    # Force text type
    chunk[TEXT_COL] = chunk[TEXT_COL].astype(str)

    # Temporary tokenization (NOT saved)
    raw_tokens = chunk[TEXT_COL].apply(tokenize_text)

    # Update pre-cleaning stats from raw tokens
    for toks in raw_tokens:
        stats["total_tokens_before"] += len(toks)
        stats["unique_tokens_before"].update(toks)
        stats["top_words_before"].update(toks)

    # Clean tokens
    chunk["tokens_clean"] = raw_tokens.apply(clean_tokens)

    # Update post-stopword stats
    for toks in chunk["tokens_clean"]:
        stats["total_tokens_after_stopwords"] += len(toks)
        stats["unique_tokens_after_stopwords"].update(toks)
        stats["top_words_after_stopwords"].update(toks)

    # Stemmed tokens
    chunk["tokens_stemmed"] = chunk["tokens_clean"].apply(stem_tokens)

    # Update stemming stats
    for toks in chunk["tokens_stemmed"]:
        stats["unique_tokens_after_stemming"].update(toks)
        stats["top_words_after_stemming"].update(toks)

    # Build clean text for later modelling
    chunk["clean_text"] = chunk["tokens_stemmed"].apply(lambda x: " ".join(x))

    # Count processed rows
    stats["n_rows"] += len(chunk)

    # Save only useful columns
    cols_to_save = []

    if LABEL_COL in chunk.columns:
        cols_to_save.append(LABEL_COL)

    cols_to_save += ["tokens_stemmed"]

    chunk_out = chunk[cols_to_save].copy()

    out_file = OUTPUT_DIR / f"chunk_{i:03d}.pkl"
    chunk_out.to_pickle(out_file)
    chunk_files.append(str(out_file))

    # Free memory
    del chunk
    del chunk_out
    del raw_tokens

Processing chunks: 0it [00:00, ?it/s]

In [7]:
summary = {
    "n_rows": stats["n_rows"],
    "n_missing_content": stats["n_missing_content"],
    "total_tokens_before": stats["total_tokens_before"],
    "total_tokens_after_stopwords": stats["total_tokens_after_stopwords"],
    "vocab_size_before": len(stats["unique_tokens_before"]),
    "vocab_size_after_stopwords": len(stats["unique_tokens_after_stopwords"]),
    "vocab_size_after_stemming": len(stats["unique_tokens_after_stemming"]),
    "reduction_rate_stopwords": 1 - (
        len(stats["unique_tokens_after_stopwords"]) / len(stats["unique_tokens_before"])
        if len(stats["unique_tokens_before"]) > 0 else 1
    ),
    "reduction_rate_stemming": 1 - (
        len(stats["unique_tokens_after_stemming"]) / len(stats["unique_tokens_after_stopwords"])
        if len(stats["unique_tokens_after_stopwords"]) > 0 else 1
    ),
    "top_100_before": stats["top_words_before"].most_common(100),
    "top_100_after_stopwords": stats["top_words_after_stopwords"].most_common(100),
    "top_100_after_stemming": stats["top_words_after_stemming"].most_common(100),
    "chunk_files": chunk_files,
}

with open("task2_summary.pkl", "wb") as f:
    pickle.dump(summary, f)

with open("task2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("rows processed:", summary["n_rows"])
print("vocabulary before:", summary["vocab_size_before"])
print("vocabulary after stopwords:", summary["vocab_size_after_stopwords"])
print("vocabulary after stemming:", summary["vocab_size_after_stemming"])
print("Saved chunks:", len(chunk_files))

rows processed: 994988
vocabulary before: 2680387
vocabulary after stopwords: 1206288
vocabulary after stemming: 1049849
Saved chunks: 100


In [8]:
#Tjek en chunk
df1 = pd.read_pickle("processed_chunks/chunk_000.pkl")
print(df1.head())

         type                                     tokens_stemmed
0   political  [plu, one, articl, googl, plu, thank, ali, alf...
1        fake  [cost, best, senat, bank, committe, jp, morgan...
2      satire  [man, awoken, coma, commit, suicid, learn, don...
3    reliable  [julia, geist, ask, draw, pictur, comput, scie...
4  conspiracy  [compil, studi, vaccin, danger, activist, post...


In [9]:
files = sorted(Path("processed_chunks").glob("chunk_*.pkl"))

for file in files[:3]:
    df = pd.read_pickle(file)
    
    print(file.name)
    print(df.head())

chunk_000.pkl
         type                                     tokens_stemmed
0   political  [plu, one, articl, googl, plu, thank, ali, alf...
1        fake  [cost, best, senat, bank, committe, jp, morgan...
2      satire  [man, awoken, coma, commit, suicid, learn, don...
3    reliable  [julia, geist, ask, draw, pictur, comput, scie...
4  conspiracy  [compil, studi, vaccin, danger, activist, post...
chunk_001.pkl
             type                                     tokens_stemmed
10000   political  [exactli, surpris, today, al, franken, announc...
10001        hate  [imagin, horrid, must, civilian, unarm, unprot...
10002  conspiracy  [let, honest, saw, one, come, govern, doubt, u...
10003   political  [silicon, valley, often, piss, internet, week,...
10004   political  [new, find, free, speech, view, colleg, studen...
chunk_002.pkl
             type                                     tokens_stemmed
20000        fake  [lijun, fu, kun, tang, chen, li, liu, xiangxin...
20001    reliabl

# Data Exploration:

In [ ]:


first_100 = files[:100]
all_types = set()
for file in tqdm(first_100):
    df = pd.read_pickle(file)
    types = df["type"].dropna().unique()
    for typ in types():
        all_types.add(typ)
    st = ", ".join(types)
    
for t in all_types:
    print(f"{t}")
print(f"number of types: {len(all_types)}")

    
    

  0%|          | 0/100 [00:00<?, ?it/s]

political, fake, satire, reliable, conspiracy, unreliable, bias, rumor, unknown, clickbait, hate, junksci
number of types: 12
political, hate, conspiracy, clickbait, reliable, fake, unreliable, rumor, bias, unknown, junksci, satire
number of types: 12
fake, reliable, clickbait, conspiracy, political, unreliable, bias, unknown, rumor, junksci, hate, satire
number of types: 12
rumor, reliable, clickbait, conspiracy, fake, political, bias, unreliable, unknown, satire, hate, junksci
number of types: 12
reliable, fake, hate, political, bias, rumor, conspiracy, clickbait, unknown, satire, unreliable, junksci
number of types: 12
unreliable, reliable, bias, unknown, political, fake, conspiracy, rumor, hate, clickbait, satire, junksci
number of types: 12
reliable, bias, conspiracy, clickbait, rumor, unreliable, fake, political, junksci, unknown, satire, hate
number of types: 12
political, fake, reliable, rumor, unknown, bias, clickbait, conspiracy, satire, junksci, unreliable, hate
number of ty

files[:60]
files[60:80]
files[80:100]